# SUPERESTELA

parair probando modelos 

## Instalacion y lectura de  datos

In [ ]:
%pip install pandas
%pip install pyarrow
%pip install fastparquet
%pip install xgboost
%pip install lightgbm
%pip install mlflow


## Normalizacion de Datos

In [ ]:
import   pandas  as pd
def normalize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia tipos de dato mixtos ANTES de construir el target o entrenar.
 
    1) Fuerza a float64 las columnas numéricas que llegaron como 'object'
       (ej. EffectiveVelo) lo que no se pueda convertir se vuelve NaN en vez de tronar el pipeline.
    2) Convierte las columnas bandera 0/1 a booleano real (maneja NaN
       tratándolos como False).
    """
    df = df.copy()
    columns = df.columns
    df["EffectiveVelo"] = pd.to_numeric(df["EffectiveVelo"], errors='coerce')
    for col in columns:
        if df[col].nunique() == 2 and set(df[col].dropna().unique()) == {0.0, 1.0}:
            #print(f"Converting column {col} to boolean")
            df[col] = df[col].astype(bool)

    #print("Columns with boolean dtype:", columns_bool)
    return  df


## Construccion de Targets

In [ ]:
RUN_VALUES = {
    "HomeRun": 1.40,
    "Triple": 1.05,
    "Double": 0.77,
    "Single": 0.47,
    "Walk": 0.31,
    "HitByPitch": 0.34,
    "Error": 0.05,
    "FieldersChoice": -0.10,
    "Out": -0.27,
    "Sacrifice": -0.10,
    "Strikeout": -0.27,
    "StrikeSwinging": -0.05,
    "StrikeCalled": -0.05,
    "FoulBallFieldable": -0.05,
    "FoulBall": -0.05,
    "BallCalled": 0.032,
    "BallinDirt": 0.032,
}
def classify_event(row) -> str | None:
    """Clasifica cada pitch en un evento discreto para mapear su run value.
 
    Basado en los valores REALES de PitchCall confirmados en el dataset:
    BallCalled, InPlay, FoulBallNotFieldable, StrikeCalled, StrikeSwinging,
    BallinDirt, FoulBall, HitByPitch, FoulBallFieldable, Undefined.
 
    
    """
    pitch_call = row.get("PitchCall")
    play_result = row.get("play_result")
    if play_result=="NeutralPlay":
        if pitch_call in ("FoulBallNotFieldable", "FoulBall"):
            return "FoulBall"
        elif pitch_call=="StrikeCalled":
            return "StrikeCalled"
        else:
            #print(f"Warning: NeutralPlay with unexpected PitchCall: {pitch_call}")
            return None

    return play_result
def build_run_value(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["event_type"] = df.apply(classify_event, axis=1)
    df["run_value"] = df["event_type"].map(RUN_VALUES)
    unmapped = df["run_value"].isna().sum()
    if unmapped:
        print(f"Aviso: {unmapped} pitches con event_type sin mapear en "
              f"RUN_VALUES: {df.loc[df['run_value'].isna(), 'event_type'].unique()}")
    df = df.dropna(subset=["run_value"])
    return df


In [ ]:

def build_whiff(df: pd.DataFrame) -> pd.DataFrame:
    """ Ya esta la columna "is_whiff"
    """
    print("length of samples with is_whiff = 1:", len(df[df["is_whiff"] == 1]))
    return df
def build_chase(df: pd.DataFrame) -> pd.DataFrame:
    """ya esta la columna "swung_outside_strike_zone"
    """
    print("length of samples with swung_outside_strike_zone = 1:", len(df[df["swung_outside_strike_zone"] == 1]))
    return df  
def build_called_strike(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ya esta la columna "is_called_strike"
    """
    print("length of samples with is_called_strike = 1:", len(df[df["is_called_strike"] == 1]))
    return df
def build_weak_contact(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega columna is_weak_contact a df, que es la probabilidad de weak contact
    """
    def is_weak_contact(row):
        exitSpeed= row.get("ExitSpeed")
        if exitSpeed is not None and exitSpeed < 71: # definimos a un contacto debil menora 71 mph
            return 1.0
        return 0.0
    df = df.copy()
    df["is_weak_contact"] = df.apply(is_weak_contact, axis=1)
    print("length of samples with is_weak_contact = 1:", len(df[df["is_weak_contact"] == 1]))
    return df


def build_groundball(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega columna groundball a df si un pitcheo  resulta en un groundball (hit_type == "GroundBall") o no.
    """
    df = df.copy()
    def is_groundball(row):
        launch_angle = row.get("Angle")
        hit_type = row.get("hit_type")
        if hit_type == "GroundBall":
            return 1.0 
        return 0.0
    df["is_groundball"] = df.apply(is_groundball, axis=1)
    print("length of samples with is_groundball = 1:", len(df[df["is_groundball"] == 1]))
    return df

def build_barrel_hard_contact(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega la columna barrel_hard_contact: 1 si el batazo fue una línea o
    fly ball (hit_type ya clasificado por Diablos) con ExitSpeed dentro del
    15% más fuerte de esta liga (percentil 85), en vez de un umbral fijo
    tipo MLB que puede no aplicar a la distribución real de LMB.
    """
    df = df.copy()

    exit_speed = pd.to_numeric(df["ExitSpeed"], errors="coerce")
    is_batted = df["hit_type"].notna() & (df["hit_type"] != "None")

    # Umbral data-driven: P85 de ExitSpeed SOLO entre bolas realmente
    # puestas en juego (no metas los NaN de pitches que no fueron bateados,
    # o el percentil sale distorsionado hacia abajo).
    threshold = exit_speed[is_batted].quantile(0.85)
    print(f"Umbral hard contact (P85 ExitSpeed en batted balls): {threshold:.1f} mph")

    is_barrel = (
        df["hit_type"].isin(["LineDrive", "FlyBall"])
        & (exit_speed >= threshold)
    )

    df["barrel_hard_contact"] = is_barrel.astype(float)
    print("length of samples with barrel_hard_contact = 1:",
          len(df[df["barrel_hard_contact"] == 1]))
    return df


def build_out_within_plate_appearance(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega la columna out_within_pa: 1 si el resultado de la aparición al
    plato fue un out del BATEADOR (strikeout, out de bola en juego, o sacrifice). Requiere que 'event_type' ya exista (corre esto DESPUÉS de
    build_run_value). Se excluye a propósito FieldersChoice (el bateador
    llega a base, no es out él) y los fouls (el turno sigue vivo)."""
    df = df.copy()

    def is_out(row):
        event = row.get("event_type")
        return 1.0 if event in ("Strikeout", "Out", "Sacrifice") else 0.0

    df["out_within_pa"] = df.apply(is_out, axis=1)
    print("length of samples with out_within_pa = 1:",
          len(df[df["out_within_pa"] == 1]))
    return df

In [ ]:
def build_xRunValue(df):
    return df

In [ ]:
# df = build_run_value(df)
# # df = build_whiff(df)
# # df = build_chase(df)
# # df = build_called_strike(df)
# # df = build_weak_contact(df)
# # df = build_groundball(df)
# # df = build_barrel_hard_contact(df)
# df = build_out_within_plate_appearance(df)


### analisis de neutral play 

In [ ]:
# df_neutral = df[df['play_result'] == 'NeutralPlay']
# print(len(df_neutral))
# df_neutral['PitchCall'].value_counts()

## submodelos para probabilidad de Targets

In [ ]:
import pandas as pd
# CONFIG
# ----------------------------------------------------------------------------
DATA_PATH = "stuff_model_df.parquet"          # <-- tu dataset real
MODELS_DIR = "models"                      # carpeta donde se guardan los .json por pitch type
NORM_PARAMS_PATH = "stuff_plus_norm_params.json"

GROUP_COL = "pitcher_anon_id"              # para el pitcher holdout
PITCH_TYPE_COL = "AutoPitchType"
YEAR_COL = "year"
ALTITUDE_COL = "altitude_category"

# Features físicas puras (SIN ubicación) - basado en tu diccionario de datos
STUFF_FEATURES = [
    "RelSpeed", "EffectiveVelo", "ZoneTime",
    "SpinRate", "SpinAxis",
    "InducedVertBreak", "HorzBreak",
    "RelHeight", "RelSide", "Extension",
    "VertRelAngle", "HorzRelAngle", "VertApprAngle", "HorzApprAngle",
]
ALTITUD_FEATURES = ["altitude_category"]
CATEGORICAL_FEATURES = ["PitcherThrows", "BatterSide"]
LOCATION_FEATURES = ["PlateLocHeight", "PlateLocSide", "in_strike_zone"]
COUNT_FEATURES = ["Balls", "Strikes","Outs"]

def build_auxiliary_targets(df: pd.DataFrame) -> pd.DataFrame:
    """Corre los  builders  """
    df = build_run_value(df)
    df = build_whiff(df)
    df = build_chase(df)
    df = build_called_strike(df)
    df = build_weak_contact(df)
    df = build_groundball(df)
    df = build_barrel_hard_contact(df)
    df = build_out_within_plate_appearance(df)
    df = build_xRunValue(df)
    return df


TARGETS_CONFIG = {
    "p_whiff": {
        "target_col": "is_whiff",
        "features": STUFF_FEATURES, #+ LOCATION_FEATURES + COUNT_FEATURES,
        "row_filter": None,
    },
    "p_chase": {
        "target_col": "swung_outside_strike_zone",
        "features": STUFF_FEATURES ,#+ LOCATION_FEATURES + COUNT_FEATURES,
        "row_filter": lambda df: df["outside_strike_zone"] == 1,
    },
    "p_called_strike": {
        "target_col": "is_called_strike",
        "features": STUFF_FEATURES ,
        "row_filter": None,  
    },
    "p_weak_contact": {
        "target_col": "is_weak_contact",
        "features": STUFF_FEATURES ,#+ LOCATION_FEATURES,
        "row_filter": lambda df: df['PitchCall'] == 'InPlay',  # noqa: E712
    },
    "p_groundball": {
        "target_col": "is_groundball",
        "features": STUFF_FEATURES,# + LOCATION_FEATURES,
        "row_filter": lambda df: df['PitchCall'] == 'InPlay',  # noqa: E712
    },
    "p_barrel_hard_contact": {
        "target_col": "barrel_hard_contact",
        "features": STUFF_FEATURES ,#+ LOCATION_FEATURES,
        "row_filter": lambda df: df['PitchCall'] == 'InPlay',  # noqa: E712
    },
    "p_out_within_pa": {
        "target_col": "out_within_pa",
        "features": STUFF_FEATURES ,#+ LOCATION_FEATURES + COUNT_FEATURES,
        "row_filter": None,
    },
}

In [ ]:
import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

In [ ]:
"""
04_train_binary_targets.py

Entrena un modelo de clasificación binaria (XGBoost) por cada target
auxiliar (P(whiff), P(chase), P(called_strike), P(weak_contact),
P(groundball), P(barrel/hard_contact), P(out_within_pa)), CADA UNO como su
propio experimento de MLflow ("stuff_plus_p_whiff", "stuff_plus_p_chase", etc.)

A diferencia de 01_train_stuff_plus.py, aquí NO se entrena un modelo por
pitch type - se entrena UN modelo por target, usando AutoPitchType como
feature categórica más (más simple, y suficiente para targets binarios;
si luego quieres separar por pitch type, es el mismo patrón que en 01).

Corre esto DESPUÉS de tener el dataset con event_type/run_value ya
construido (build_run_value) y los 7 targets auxiliares ya construidos
(build_auxiliary_targets).
"""

import json
import os

import pandas as pd
import xgboost as xgb
import mlflow
import mlflow.xgboost
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss
import matplotlib.pyplot as plt

RANDOM_STATE = 42
MIN_ROWS_TO_TRAIN = 500   # si el subset filtrado queda muy chico, se salta


def train_binary_target(df: pd.DataFrame, target_name: str, config: dict):
    target_col = config["target_col"]
    features = config["features"]
    row_filter = config["row_filter"]
    categorical = [c for c in CATEGORICAL_FEATURES if c in features]

    sub = df if row_filter is None else df[row_filter(df)].copy()
    sub = sub.dropna(subset=[target_col, GROUP_COL])
    if len(sub) < MIN_ROWS_TO_TRAIN:
        print(f"[{target_name}] Saltando: solo {len(sub)} filas después del filtro.")
        return None

    X = sub[features]
    y = sub[target_col].astype(int)

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
    train_idx, test_idx = next(splitter.split(sub, groups=sub[GROUP_COL]))
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    n_pos, n_neg = y_train.sum(), len(y_train) - y_train.sum()
    scale_pos_weight = n_neg / max(n_pos, 1)

    params = {
        "objective": "binary:logistic",
        "eval_metric": ["logloss", "aucpr"],
        "eta": 0.05,
        "max_depth": 6,
        "min_child_weight": 50,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "tree_method": "hist",
        "scale_pos_weight": scale_pos_weight,
        "seed": RANDOM_STATE,
    }

    # Un experimento de MLflow POR TARGET, como pediste
    mlflow.set_experiment(f"stuff_plus_{target_name}")
    with mlflow.start_run(run_name=target_name):
        mlflow.log_params(params)
        mlflow.log_param("target_col", target_col)
        mlflow.log_param("features", features)
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))
        mlflow.log_param("positive_rate_train", float(y_train.mean()))
        mlflow.log_param("positive_rate_test", float(y_test.mean()))

        dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)

        model = xgb.train(
            params, dtrain,
            num_boost_round=500,
            evals=[(dtrain, "train"), (dtest, "valid")],
            early_stopping_rounds=30,
            verbose_eval=50,
        )

        preds = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))

        auc = roc_auc_score(y_test, preds)
        aucpr = average_precision_score(y_test, preds)
        ll = log_loss(y_test, preds)
        # Baseline: siempre predecir la tasa base (positive_rate_train)
        baseline_preds = [y_train.mean()] * len(y_test)
        baseline_ll = log_loss(y_test, baseline_preds)

        mlflow.log_metric("auc", auc)
        mlflow.log_metric("aucpr", aucpr)
        mlflow.log_metric("log_loss", ll)
        mlflow.log_metric("baseline_log_loss", baseline_ll)
        mlflow.log_metric("log_loss_improvement_pct", 100 * (baseline_ll - ll) / baseline_ll)
        mlflow.log_metric("best_iteration", model.best_iteration)

        fig, ax = plt.subplots(figsize=(6, 8))
        xgb.plot_importance(model, ax=ax, max_num_features=15, importance_type="gain")
        plt.title(f"Feature importance - {target_name}")
        plt.tight_layout()
        fig_path = f"feature_importance_{target_name}.png"
        fig.savefig(fig_path)
        mlflow.log_artifact(fig_path)
        plt.close(fig)

        mlflow.xgboost.log_model(model, artifact_path="model")

        os.makedirs(MODELS_DIR, exist_ok=True)
        model_path = os.path.join(MODELS_DIR, f"model_{target_name}.json")
        model.save_model(model_path)

        print(f"[{target_name}] AUC={auc:.4f}  AUCPR={aucpr:.4f}  "
              f"logloss={ll:.4f} (baseline={baseline_ll:.4f})  "
              f"-> guardado en {model_path}")

        return model


## MAIN  Entrenando  targets  

In [ ]:
print("Cargando dataset y construyendo targets...")
df = pd.read_parquet(DATA_PATH)        # build_target=True: crea event_type/run_value
df=normalize_dtypes(df)
df = build_auxiliary_targets(df)       # crea los 7 targets p_*

In [ ]:
# print("entrnando targets")
# results = {}
# for target_name, config in TARGETS_CONFIG.items():
#     print(f"\n{'='*70}\nEntrenando: {target_name}\n{'='*70}")
#     model = train_binary_target(df, target_name, config)
#     results[target_name] = "OK" if model is not None else "SALTADO"

# print("\n=== RESUMEN ===")
# for target_name, status in results.items():
#     print(f"{target_name}: {status}")

### añadiendo la probabilidad al df

In [ ]:
import os
import glob
import xgboost as xgb
import pandas as pd

MODELS_DIR = "models"
model_files = sorted(glob.glob(os.path.join(MODELS_DIR, "model_p_*.json")))

print(f"Se encontraron {len(model_files)} modelos guardados en '{MODELS_DIR}':")

for model_path in model_files:
    # Extraer el nombre del target (ej: model_p_whiff.json -> p_whiff)
    target_name = os.path.basename(model_path).replace("model_", "").replace(".json", "")
    pred_col = f"pred_{target_name}"
    
    # Cargar el modelo Booster
    bst = xgb.Booster()
    bst.load_model(model_path)
    
    # Obtener exactamente las features con las que fue entrenado ese modelo
    model_features = bst.feature_names
    print("ejores",model_features)
    # Crear DMatrix y generar predicciones
    dmat = xgb.DMatrix(df[model_features], enable_categorical=True)
    df[pred_col] = bst.predict(dmat)
    
    print(f" -> {pred_col:<25} | Media: {df[pred_col].mean():.4f} | Rango: [{df[pred_col].min():.3f}, {df[pred_col].max():.3f}]")

print("\n¡Todas las columnas de probabilidad p_* han sido agregadas exitosamente a df!")


### analisis targets

In [ ]:
whiff_counts = df[df['is_whiff'] == True]['AutoPitchType'].value_counts()
print("cual genera mas whiff")
print(whiff_counts)
print("cual genera mas chase")
chase_counts = df[df['swung_outside_strike_zone'] == True]['AutoPitchType'].value_counts()
print(chase_counts)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon

def plot_heat_map_strike_zone_by_column(df, column, pitcher_id=None, pitch_type=None, gridsize=25, min_cnt=2, invert_side=True,title=None
):
    """
    Grafica un mapa de calor en la zona de strike para cualquier columna.
    
    - Si la columna es binaria (0/1): Filtra los 1s y muestra densidad de ocurrencia.
    - Si la columna es continua/probabilidad: Muestra la probabilidad promedio en cada zona.
    
    Parámetros:
    -----------
    df : pd.DataFrame con 'PlateLocSide' y 'PlateLocHeight'.
    column : str, nombre de la columna a graficar (ej. 'is_whiff', 'pred_p_whiff', 'Stuff_Plus').
    pitcher_id : str (opcional), filtra a un lanzador específico (ej. 'pitcher_00619').
    pitch_type : str (opcional), filtra a un tipo de pitcheo (ej. 'Slider', 'Four-Seam').
    gridsize : int, resolución de los hexágonos del heatmap (default: 25).
    min_cnt : int, mínimo de lanzamientos en un hexágono para mostrar color (default: 2).
    invert_side : bool, si True invierte el eje X para vista desde el catcher/umpire (default: True).
    """
    required = ['PlateLocSide', 'PlateLocHeight', column]
    for col in required:
        if col not in df.columns:
            raise ValueError(f"La columna '{col}' no existe en el DataFrame.")

    # 1. Filtros opcionales de pitcher y tipo de lanzamiento
    data = df.dropna(subset=['PlateLocSide', 'PlateLocHeight', column]).copy()
    filter_desc = []
    
    if pitcher_id is not None:
        data = data[data['pitcher_anon_id'] == pitcher_id]
        filter_desc.append(f"Pitcher: {pitcher_id}")
    if pitch_type is not None:
        data = data[data['AutoPitchType'] == pitch_type]
        filter_desc.append(f"Tipo: {pitch_type}")
        
    if len(data) == 0:
        print("No hay datos disponibles para los filtros seleccionados.")
        return

    # Ajuste de perspectiva (Catchers view: PlateLocSide * -1)
    x = data['PlateLocSide'] * (-1 if invert_side else 1)
    y = data['PlateLocHeight']
    vals = data[column]

    # 2. Detectar si es columna binaria o continua / probabilidad
    unique_vals = set(vals.dropna().unique())
    is_binary = unique_vals.issubset({0, 1, 0.0, 1.0, True, False}) and len(unique_vals) <= 2

    fig, ax = plt.subplots(figsize=(8, 9), facecolor='white')

    # =========================================================================
    # CASO A: Columna Binaria (ej. is_whiff, is_called_strike)
    # =========================================================================
    if is_binary:
        mask_pos = (vals == 1) | (vals == True)
        x_filtered = x[mask_pos]
        y_filtered = y[mask_pos]
        n_pos = len(x_filtered)
        pct = (n_pos / max(len(data), 1)) * 100
        
        hb = ax.hexbin(x_filtered, y_filtered, gridsize=gridsize, cmap='YlOrRd', mincnt=1,edgecolors='none',zorder=3)
        cbar = plt.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(f'Frecuencia de {column} == 1 (Conteo)', fontsize=11, fontweight='semibold')
        mode_label = f"Eventos: {n_pos:,} / {len(data):,} ({pct:.1f}%)"

    # =========================================================================
    # CASO B: Columna Continua / Probabilidad (ej. pred_p_whiff, Stuff_Plus)
    # =========================================================================
    else:
        hb = ax.hexbin(x, y, C=vals, reduce_C_function=np.mean, gridsize=gridsize, cmap='coolwarm', mincnt=min_cnt,edgecolors='none',zorder=3)
        cbar = plt.colorbar(hb, ax=ax, fraction=0.046, pad=0.04)
        
        # Etiqueta de la barra según si es probabilidad o índice
        if vals.max() <= 1.0 and vals.min() >= 0.0:
            cbar.set_label(f'Probabilidad Promedio de {column}', fontsize=11, fontweight='semibold')
            cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f"{val*100:.1f}%"))
            mode_label = f"Prob. Global: {vals.mean()*100:.1f}% (n={len(data):,})"
        else:
            cbar.set_label(f'Valor Promedio de {column}', fontsize=11, fontweight='semibold')
            mode_label = f"Media: {vals.mean():.2f} (n={len(data):,})"

    # =========================================================================
    # 3. Dibujar Zona de Strike y Home Plate
    # =========================================================================
    zone_width = 1.66       # 17 pulgadas + radio bola en pies
    zone_left = -zone_width / 2
    top_of_zone = 3.5
    bottom_of_zone = 1.6
    zone_height = top_of_zone - bottom_of_zone

    # Rectángulo exterior de la zona
    strike_zone = Rectangle((zone_left, bottom_of_zone),zone_width,zone_height,facecolor='none',
        edgecolor='black',linewidth=2.5,zorder=5,label='Zona de Strike')
    ax.add_patch(strike_zone)

    # Cuadrícula interna 3x3
    w_third = zone_width / 3
    h_third = zone_height / 3
    for i in range(1, 3):
        # Líneas verticales
        ax.plot([zone_left + i * w_third, zone_left + i * w_third],
                [bottom_of_zone, top_of_zone], color='black', linestyle=':', linewidth=1.2, zorder=5)
        # Líneas horizontales
        ax.plot([zone_left, zone_left + zone_width],
                [bottom_of_zone + i * h_third, bottom_of_zone + i * h_third], color='black', linestyle=':', linewidth=1.2, zorder=5)

    # Pentágono del Home Plate (vista aérea colocada abajo como referencia)
    plate_y = 0.5
    plate_pts = np.array([
        [-0.708, plate_y],
        [0.708, plate_y],
        [0.708, plate_y - 0.2],
        [0, plate_y - 0.4],
        [-0.708, plate_y - 0.2]
    ])
    home_plate = Polygon(plate_pts, closed=True, facecolor='lightgray', edgecolor='black', linewidth=1.5, zorder=4)
    ax.add_patch(home_plate)

    # =========================================================================
    # 4. Estilo, Ejes y Títulos
    # =========================================================================
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, zorder=2)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 5.0)
    ax.set_aspect('equal')
    ax.set_xlabel('PlateLocSide (pies) - Perspectiva del Catcher', fontsize=11)
    ax.set_ylabel('PlateLocHeight (pies)', fontsize=11)
    ax.grid(alpha=0.15, linestyle='--')

    # Título principal y subtítulo
    filt_txt = f" [{', '.join(filter_desc)}]" if filter_desc else ""
    plot_title = title if title else f"Distribución de {column} en Zona de Strike{filt_txt}"
    ax.set_title(f"{plot_title}\n{mode_label}", fontsize=12, fontweight='bold', pad=12)

    plt.tight_layout()
    plt.show()


In [ ]:
df.keys()

In [ ]:
# plot_heat_map_strike_zone_by_column(df.head(100000), "outside_strike_zone")
# plot_heat_map_strike_zone_by_column(df.head(100000), "is_whiff",pitch_type='Four-Seam')
# plot_heat_map_strike_zone_by_column(df.head(100000), "is_whiff",pitch_type="Slider")
# plot_heat_map_strike_zone_by_column(df.head(100000), "home_run")
# plot_heat_map_strike_zone_by_column(df.head(100000), "is_weak_contact")
# plot_heat_map_strike_zone_by_column(df.head(100000), "barrel_hard_contact")
plot_heat_map_strike_zone_by_column(df.head(100000), "pred_p_called_strike")



In [ ]:
tipoPitcheo='Changeup'
plot_heat_map_strike_zone_by_column(df, "is_whiff",pitch_type=tipoPitcheo)
plot_heat_map_strike_zone_by_column(df, "pred_p_whiff",pitch_type=tipoPitcheo)

plot_heat_map_strike_zone_by_column(df, "swung_outside_strike_zone",pitch_type=tipoPitcheo)
plot_heat_map_strike_zone_by_column(df, "pred_p_chase",pitch_type=tipoPitcheo)




In [ ]:
plot_heat_map_strike_zone_by_column(df, "pred_p_barrel_hard_contact",pitch_type=tipoPitcheo)
plot_heat_map_strike_zone_by_column(df, "barrel_hard_contact",pitch_type=tipoPitcheo)


## añadir el xRunValue(el target Bueno) a los pitches con modelo

In [ ]:

# 1. Asignación de pesos lineales (Run Values del bateador por evento)
# Un valor negativo favorece al PITCHER (evita carreras).
# Un valor positivo favorece al BATEADOR (anota carreras).
W_WHIFF          = -0.060   # Strike tirándole / whiff
W_CALLED_STRIKE  = 0   # Strike cantado
W_WEAK_CONTACT   = -0.120   # Contacto débil (casi siempre es out)
W_GROUNDBALL     = 0.0   # Rolling (alta probabilidad de out)
W_HARD_CONTACT   = +0.380   # Macanazo / Hard hit (frecuentemente hit o extrabase)
W_OUT_PA         = 0   # Out consumado en la aparición

# 2. Calcular el Expected Run Value (xRV) por pitcheo
df["xRV"] = (
    df["pred_p_whiff"]                 * W_WHIFF +
    df["pred_p_called_strike"]         * W_CALLED_STRIKE +
    df["pred_p_weak_contact"]          * W_WEAK_CONTACT +
    df["pred_p_groundball"]            * W_GROUNDBALL +
    df["pred_p_barrel_hard_contact"]   * W_HARD_CONTACT +
    df["pred_p_out_within_pa"]         * W_OUT_PA
)

In [ ]:
tipoPitcheo='Four-Seam'
plot_heat_map_strike_zone_by_column(df, "xRV",pitch_type=tipoPitcheo)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_heat_map_2d(df,x_col,y_col,target_col,pitcher_id=None,pitch_type=None,gridsize=30,min_cnt=3,cmap='coolwarm',clean_outliers=True,title=None):
    """
    Genera un mapa de calor 2D cruzando dos métricas físicas (x_col, y_col)
    y coloreando cada región según el valor promedio de target_col.
    
    Parámetros:
    -----------
    df : pd.DataFrame con los datos.
    x_col : str, nombre de la columna para el eje X (ej. 'SpinRate', 'Extension', 'HorzBreak').
    y_col : str, nombre de la columna para el eje Y (ej. 'InducedVertBreak', 'RelSpeed').
    target_col : str, columna objetivo para el color (ej. 'is_whiff', 'pred_p_whiff', 'Stuff_Plus').
    pitcher_id : str (opcional), filtra por ID de pitcher (ej. 'pitcher_00619').
    pitch_type : str (opcional), filtra por tipo de pitcheo (ej. 'Four-Seam', 'Slider').
    gridsize : int, resolución de los hexágonos en el plano (default: 30).
    min_cnt : int, lanzamientos mínimos por hexágono para pintarlo (default: 3).
    cmap : str, paleta de colores ('coolwarm', 'viridis', 'plasma', 'YlOrRd', etc.).
    clean_outliers : bool, recorta percentiles extremos (1% y 99%) para evitar ruido de TrackMan.
    """
    cols_needed = [x_col, y_col, target_col]
    for c in cols_needed:
        if c not in df.columns:
            raise ValueError(f"La columna '{c}' no existe en el DataFrame.")

    # 1. Filtrar datos limpios y filtros opcionales
    data = df.dropna(subset=cols_needed).copy()
    filter_tags = []
    
    if pitcher_id is not None:
        data = data[data['pitcher_anon_id'] == pitcher_id]
        filter_tags.append(f"Pitcher: {pitcher_id}")
    if pitch_type is not None:
        data = data[data['AutoPitchType'] == pitch_type]
        filter_tags.append(f"Tipo: {pitch_type}")
        
    if len(data) == 0:
        print("No hay suficientes datos para los filtros seleccionados.")
        return

    # Limpieza de valores atípicos extremos para que los ejes queden bien encuadrados
    if clean_outliers:
        q_low_x, q_hi_x = data[x_col].quantile(0.005), data[x_col].quantile(0.995)
        q_low_y, q_hi_y = data[y_col].quantile(0.005), data[y_col].quantile(0.995)
        data = data[(data[x_col] >= q_low_x) & (data[x_col] <= q_hi_x) &
                    (data[y_col] >= q_low_y) & (data[y_col] <= q_hi_y)]

    x = data[x_col].values
    y = data[y_col].values
    vals = data[target_col].values

    # 2. Detectar si el target es binario (0/1) o continuo
    unique_vals = set(pd.Series(vals).dropna().unique())
    is_binary = unique_vals.issubset({0, 1, 0.0, 1.0, True, False}) and len(unique_vals) <= 2

    fig, ax = plt.subplots(figsize=(10, 7.5), facecolor='white')

    # 3. Graficar Hexbin 2D con agregación por promedio (np.mean)
    hb = ax.hexbin(x, y, C=vals.astype(float), reduce_C_function=np.mean, gridsize=gridsize, cmap=cmap, mincnt=min_cnt,edgecolors='gray',linewidths=0.2,zorder=3)

    # 4. Barra de color y leyenda según el tipo de target
    cbar = plt.colorbar(hb, ax=ax, fraction=0.046, pad=0.03)
    
    if is_binary:
        cbar.set_label(f'Tasa Promedio de {target_col} (% de éxito)', fontsize=11, fontweight='semibold')
        cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f"{val*100:.1f}%"))
        stat_text = f"Tasa Global {target_col}: {vals.mean()*100:.1f}%\nMuestra: {len(data):,} pitcheos"
    elif vals.max() <= 1.0 and vals.min() >= 0.0:
        cbar.set_label(f'Probabilidad Promedio de {target_col}', fontsize=11, fontweight='semibold')
        cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda val, _: f"{val*100:.1f}%"))
        stat_text = f"Prob. Promedio: {vals.mean()*100:.1f}%\nMuestra: {len(data):,} pitcheos"
    else:
        cbar.set_label(f'Valor Promedio de {target_col}', fontsize=11, fontweight='semibold')
        stat_text = f"Media {target_col}: {vals.mean():.2f}\nMuestra: {len(data):,} pitcheos"

    # 5. Líneas de referencia (medias de X y Y)
    ax.axvline(x.mean(), color='black', linestyle=':', linewidth=1.2, alpha=0.6, label=f'Media {x_col}: {x.mean():.1f}')
    ax.axhline(y.mean(), color='black', linestyle=':', linewidth=1.2, alpha=0.6, label=f'Media {y_col}: {y.mean():.1f}')
    
    # Si alguna métrica es quiebre horizontal/vertical, marcar el 0 de quiebre neutro
    if 'break' in x_col.lower():
        ax.axvline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.5)
    if 'break' in y_col.lower():
        ax.axhline(0, color='red', linestyle='--', linewidth=0.8, alpha=0.5)

    # 6. Estética y títulos
    ax.set_xlabel(f"{x_col}", fontsize=11, fontweight='semibold')
    ax.set_ylabel(f"{y_col}", fontsize=11, fontweight='semibold')
    ax.grid(alpha=0.25, linestyle='--')
    ax.legend(loc='upper right', framealpha=0.9, fontsize=9)

    tag_str = f" [{', '.join(filter_tags)}]" if filter_tags else ""
    plot_title = title if title else f"Mapa de Calor 2D: {x_col} vs {y_col}\n(Color = {target_col}){tag_str}"
    ax.set_title(plot_title, fontsize=13, fontweight='bold', pad=12)

    # Caja informativa en la esquina inferior izquierda
    ax.text(
        0.03, 0.04, stat_text,
        transform=ax.transAxes,
        fontsize=9,
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.85, edgecolor='lightgray'),
        zorder=4
    )

    plt.tight_layout()
    plt.show()


In [ ]:
plot_heat_map_2d(
df, 
x_col="SpinRate", 
y_col="InducedVertBreak", 
target_col="pred_p_whiff",
pitch_type="Four-Seam"
)


In [ ]:
plot_heat_map_2d(
    df, 
    x_col="Extension", 
    y_col="RelSpeed", 
    target_col="pred_p_whiff",
    cmap="plasma"
)


In [ ]:
df.keys()

In [ ]:
plot_heat_map_2d(
    df, 
    x_col="HorzBreak", 
    y_col="InducedVertBreak", 
    target_col="is_whiff",
    pitch_type="Slider"
)


## entrenar stuffplus

In [ ]:
print(df[df['pitcher_anon_id'] == 'pitcher_00619']['Stuff_Plus'].mean())


## Enntrenamiento  de modelo Stuff +

## funciones extras


In [ ]:
### import pandas as pd
import numpy as np

# ============================================================================
# 1. NORMALIZAR STUFF+ POR TIPO DE PITCHEO (Estilo FanGraphs)
# ============================================================================
def normalize_by_pitch_type(df_data):
    df_res = df_data.copy()
    
    # Estandarizar xRV dentro de cada AutoPitchType
    # (Menor xRV = mejor pitcheo, por eso el signo negativo)
    stats = df_res.groupby("AutoPitchType")["xRV"].agg(["mean", "std"]).reset_index()
    stats.columns = ["AutoPitchType", "pt_mean", "pt_std"]
    
    df_res = df_res.merge(stats, on="AutoPitchType", how="left")
    df_res["Stuff_Plus_PitchType"] = 100 - 10 * ((df_res["xRV"] - df_res["pt_mean"]) / df_res["pt_std"].replace(0, 1))
    
    return df_res

df = normalize_by_pitch_type(df)

# ============================================================================
# 2. FUNCIÓN PARA OBTENER EL ARSENAL DE CUALQUIER PITCHER
# ============================================================================
def get_pitcher_arsenal(df_data, pitcher_id, min_pitches=10):
    p_df = df_data[df_data["pitcher_anon_id"] == pitcher_id]
    if len(p_df) == 0:
        print(f"Pitcher {pitcher_id} no encontrado en el dataset.")
        return None
    
    total_p = len(p_df)
    
    # Agrupar por tipo de lanzamiento
    arsenal = (
        p_df.groupby("AutoPitchType")
        .agg(
            n_pitches=("PitchUID", "count"),
            velo=("RelSpeed", "mean"),
            ivb=("InducedVertBreak", "mean"),
            hb=("HorzBreak", "mean"),
            spin=("SpinRate", "mean"),
            whiff_pct=("pred_p_whiff", "mean"),
            stuff_plus_pitch=("Stuff_Plus_PitchType", "mean"),   # Calificado vs su tipo
            stuff_plus_global=("Stuff_Plus", "mean")             # Calificado global
        )
        .reset_index()
    )
    
    # Filtrar lanzamientos con muestra representativa
    arsenal = arsenal[arsenal["n_pitches"] >= min_pitches].copy()
    arsenal["usage_pct"] = (arsenal["n_pitches"] / total_p) * 100
    
    # Ordenar por frecuencia de uso
    arsenal = arsenal.sort_values(by="usage_pct", ascending=False)
    
    # Calcular Stuff+ Overall del pitcher ponderado por uso
    overall_stuff = (arsenal["stuff_plus_pitch"] * (arsenal["n_pitches"] / arsenal["n_pitches"].sum())).sum()
    
    print(f"\n=======================================================")
    print(f" ARSENAL DE {pitcher_id}  |  Total Pitches: {total_p}  |  Overall Stuff+: {overall_stuff:.1f}")
    print(f"=======================================================")
    
    # Formato limpio para presentar
    display_df = arsenal[[
        "AutoPitchType", "usage_pct", "n_pitches", "velo", "ivb", "hb", 
        "whiff_pct", "stuff_plus_pitch"
    ]].copy()
    
    display_df["usage_pct"] = display_df["usage_pct"].map("{:.1f}%".format)
    display_df["velo"] = display_df["velo"].map("{:.1f} mph".format)
    display_df["ivb"] = display_df["ivb"].map("{:+.1f}\"".format)
    display_df["hb"] = display_df["hb"].map("{:+.1f}\"".format)
    display_df["whiff_pct"] = (display_df["whiff_pct"] * 100).map("{:.1f}%".format)
    display_df["stuff_plus_pitch"] = display_df["stuff_plus_pitch"].round(1)
    
    display(display_df)
    return arsenal
 